## 1. Reconstrucción de la tabla completa

La Fase 1 dejó la unión de las 46 hojas por número de bloque solo en el notebook del EDA, sin guardarla como archivo. Para que el preprocesamiento sea reproducible desde los Excel originales, esa unión se rehace con la función `construir_tabla_completa` de `src/preprocessing_pipeline.py`, y se controla que coincida con la del EDA.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.append("../src")
from auxiliar_functions import tabla_simple
from preprocessing_pipeline import construir_tabla_completa

import importlib
import preprocessing_pipeline

importlib.reload(preprocessing_pipeline)    # vuelve a leer el archivo de src/


<module 'preprocessing_pipeline' from 'C:\\Users\\Asus\\Desktop\\Automatizaciones\\Materias Facultad\\Tercer Año - Segundo Cuatrimestre\\Laboratorio de Datos II\\Trabajo Práctico Final LD2\\notebooks\\../src\\preprocessing_pipeline.py'>

In [2]:
from preprocessing_pipeline import cargar_tabla_completa
tabla_completa = cargar_tabla_completa(Path("../data"))   # la primera vez tarda ~6 min; después, segundos

control = pd.DataFrame({"Valor": {
    "Días (filas)": len(tabla_completa),
    "Columnas": tabla_completa.shape[1],
    "Días válidos": int(tabla_completa["valido"].sum()),
    "Filas con objetivo": int(tabla_completa["objetivo"].notna().sum()),
    "Columnas de las otras hojas": int(sum(" | " in str(c) for c in tabla_completa.columns)),
}})
print("Control de la tabla completa (debe coincidir con el EDA: 1.212 días, 453 columnas, 1.187 válidos, 1.186 con objetivo, 420 de otras hojas)")
display(tabla_simple(control, precision=0))


Control de la tabla completa (debe coincidir con el EDA: 1.212 días, 453 columnas, 1.187 válidos, 1.186 con objetivo, 420 de otras hojas)


,Valor
Días (filas),1212
Columnas,453
Días válidos,1187
Filas con objetivo,1186
Columnas de las otras hojas,420


## 2. División temporal en entrenamiento, validación y prueba

Antes de tomar cualquier decisión de limpieza o de selección se divide la tabla por orden de fecha, sin mezclar las filas: el 70 % más antiguo (848 días, hasta el 28/10/2022) para entrenamiento, el 15 % siguiente (182 días) para validación y el 15 % más reciente (182 días, desde el 28/04/2023) para prueba. Mezclar las filas, como hace `train_test_split`, dejaría días futuros dentro del entrenamiento; con este corte, cada criterio se mide solo con entrenamiento y la prueba se toca una única vez, al final. El corte de entrenamiento coincide con el que se usó en el EDA, así que las correlaciones y los criterios ya calculados siguen valiendo. Como el objetivo de cada día es el consumo del siguiente, el último día de cada tramo comparte una fila con el primero del tramo que sigue, un solapamiento de efecto despreciable.


In [3]:
from preprocessing_pipeline import dividir_por_fecha

X_train, X_val, X_test = dividir_por_fecha(tabla_completa)

resumen = pd.DataFrame({
    nombre: {"Días": len(parte),
             "Desde": parte["DIA"].iloc[0].strftime("%d/%m/%Y"),
             "Hasta": parte["DIA"].iloc[-1].strftime("%d/%m/%Y"),
             "Válidos con objetivo": int((parte["valido"] & parte["objetivo"].notna()).sum())}
    for nombre, parte in {"X_train": X_train, "X_val": X_val, "X_test": X_test}.items()}).T
print("División temporal de la tabla completa (70 % / 15 % / 15 %, sin mezclar las filas)")
display(tabla_simple(resumen, precision=0))


División temporal de la tabla completa (70 % / 15 % / 15 %, sin mezclar las filas)


,Días,Desde,Hasta,Válidos con objetivo
X_train,848,01/07/2020,28/10/2022,817
X_val,182,29/10/2022,27/04/2023,172
X_test,182,28/04/2023,26/10/2023,176


## 3. Limpieza del dataset, paso 1: columnas descartadas por ADR 0006

In [4]:
from auxiliar_functions import marcar_columnas_descartables, perfilar_columnas, UMBRAL_MUCHOS_FALTANTES, UMBRAL_CASI_CONSTANTE

# Columnas que no son candidatas a variable: fecha, hora, controles del día y el objetivo
COLUMNAS_DE_CONTROL = ["DIA", "HORA", "bloque", "filas", "archivo", "completo", "plausible",
                       "lectura_inicial", "mayor_incremento", "menor_incremento",
                       "tipo_falla", "valido", "consumo_dia", "objetivo"]
candidatas = [c for c in X_train.columns if c not in COLUMNAS_DE_CONTROL]

# El perfil se mide SOLO con entrenamiento
perfil = marcar_columnas_descartables(perfilar_columnas(X_train[candidatas]))
columnas_conservadas = [c for c, se_conserva in zip(candidatas, perfil["Se conserva"]) if se_conserva]

print("Resultado de los criterios del ADR 0006, medidos solo con X_train")
resumen_columnas = pd.DataFrame({"Columnas": {
    "Candidatas": len(perfil),
    "Con más de 40 % de faltantes": int(perfil["Muchos faltantes"].sum()),
    "Casi constantes (95 % o más igual)": int(perfil["Casi constante"].sum()),
    "Se conservan": len(columnas_conservadas)}})
display(tabla_simple(resumen_columnas, precision=0))

# La misma lista se aplica a las tres partes (las columnas de control se mantienen)
columnas_a_mantener = [c for c in X_train.columns if c in COLUMNAS_DE_CONTROL or c in columnas_conservadas]
X_train = X_train[columnas_a_mantener].copy()
X_val = X_val[columnas_a_mantener].copy()
X_test = X_test[columnas_a_mantener].copy()

print("Forma de cada parte después de descartar columnas (filas, columnas)")
display(tabla_simple(pd.DataFrame({"Filas": {n: p.shape[0] for n, p in {"X_train": X_train, "X_val": X_val, "X_test": X_test}.items()},
                                   "Columnas": {n: p.shape[1] for n, p in {"X_train": X_train, "X_val": X_val, "X_test": X_test}.items()}}), precision=0))


Resultado de los criterios del ADR 0006, medidos solo con X_train


,Columnas
Candidatas,439
Con más de 40 % de faltantes,142
Casi constantes (95 % o más igual),95
Se conservan,262


Forma de cada parte después de descartar columnas (filas, columnas)


,Filas,Columnas
X_train,848,276
X_val,182,276
X_test,182,276


### Paso 2: días inválidos

Los días marcados como inválidos (17 en `X_train`, 5 en `X_val` y 3 en `X_test`; ADR 0004 y 0005) se detectaron mirando solo el contador de `Frio (Kw)` de Consolidado EE. Para decidir qué columnas se anulan en esos días se comparó, solo con `X_train`, cuántos valores caen fuera del rango normal (percentiles 1 a 99 de los días válidos): Consolidado EE llega a 10,5 % en días inválidos contra 2,1 % en días válidos, mientras que el resto de las hojas se parece a lo normal (entre 0 % y 5,9 %, con pocos días como base).

**Decisión:** en los días inválidos se ponen en `NaN` las 19 columnas de Consolidado EE y las 18 razones "EE ... / Hl" de Consolidado KPI (derivadas de esos mismos kW). Las demás columnas se conservan; si alguna tiene valores absurdos, se trata en el paso siguiente. No se imputa nada y las filas se conservan (ADR 0004). El detalle queda en el ADR 0008.
La justificación que pensamos es la siguiente: si en un día el sensor falló o cortó antes, entonces las otras variables derivadas de datos medidos por ese sensor también son poco confiables.

In [5]:
# Columnas derivadas del contador de energía que falló: Consolidado EE y las razones "EE ... / Hl" del KPI
columnas_de_energia = [c for c in X_train.columns
                       if c not in COLUMNAS_DE_CONTROL
                       and (" | " not in c or c.startswith("Consolidado KPI | EE"))]

def anular_dias_invalidos(parte, columnas):
    """Pone vacío (NaN) las columnas indicadas en los días marcados como no válidos."""
    parte = parte.copy()
    parte.loc[~parte["valido"], columnas] = float("nan")
    return parte

X_train = anular_dias_invalidos(X_train, columnas_de_energia)
X_val = anular_dias_invalidos(X_val, columnas_de_energia)
X_test = anular_dias_invalidos(X_test, columnas_de_energia)

print(f"Columnas anuladas en los días inválidos: {len(columnas_de_energia)}")


Columnas anuladas en los días inválidos: 37


### Paso 3: valores absurdos e irreales

Se anulan con 'NaN' las celdas cuyos valores no pueden ser reales. No se corrigen ni se recortan, porque un valor recortado a un percentil parecería un dato verdadero y no lo es (mismo criterio que el ADR 0004).

**Regla** (límites calculados solo con `X_train` y aplicados igual a `X_val` y `X_test`). Una celda es absurda si:
1. es **negativa**, salvo en las columnas de temperatura (donde un valor negativo es posible), o
2. supera **5 veces el percentil 99** de su columna. Si ese percentil es 0 (columnas intermitentes, como los hectolitros de una marca que se produce pocos días), no se aplica límite.

**Por qué 5 veces:** el consumo de `Frio` varía 2,2 veces entre enero y julio, así que ninguna variación estacional real se acerca a ese factor. Entre 5 y 10 veces el percentil 99 hay 75 celdas en `X_train`; de las 66 que se pudieron comparar con los 3 días anteriores y los 3 posteriores, todas son picos aislados (de 7,6 a 1.890 veces la mediana de sus vecinos), no niveles altos sostenidos. Por encima de 10 veces se cumple lo mismo (132 de 138).

**Qué no se hace:** los valores altos pero creíbles (entre el percentil 99 y 5 veces ese valor) se dejan como están; si hace falta winsorizarlos se decide al elegir el modelo. Tampoco eliminamos filas: una fila con una sola celda absurda conserva sus otras columnas, y los días no se borran para no romper los rezagos del paso de variables nuevas. Detalle y alternativas descartadas en el ADR 0008.


In [6]:
columnas_de_medicion = [c for c in X_train.columns if c not in COLUMNAS_DE_CONTROL]   # las 262 columnas que no son de control

FACTOR_SOBRE_P99 = 5     # una celda es absurda si supera 5 veces el percentil 99 de su columna (medido en X_train)
limite_superior = FACTOR_SOBRE_P99 * X_train[columnas_de_medicion].apply(pd.to_numeric, errors="coerce").quantile(0.99)
limite_superior = limite_superior.where(limite_superior > 0)     # percentil 99 igual a 0: sin límite (columna intermitente)
columnas_de_temperatura = [c for c in columnas_de_medicion if "Temp" in c]

def marcar_valores_absurdos(parte, columnas, limite_superior, columnas_de_temperatura):
    """Devuelve dos tablas de verdadero/falso: celdas negativas (salvo temperaturas) y celdas por encima del límite."""
    valores = parte[columnas].apply(pd.to_numeric, errors="coerce")
    negativas = (valores < 0) & ~valores.columns.isin(columnas_de_temperatura)
    muy_altas = valores > limite_superior                    # comparar con NaN da falso: no se marca
    return negativas, muy_altas

def anular_valores_absurdos(parte, columnas, limite_superior, columnas_de_temperatura):
    """Pone vacío (NaN) las celdas marcadas como absurdas; el resto de la tabla queda igual."""
    parte = parte.copy()
    negativas, muy_altas = marcar_valores_absurdos(parte, columnas, limite_superior, columnas_de_temperatura)
    parte[columnas] = parte[columnas].apply(pd.to_numeric, errors="coerce").mask(negativas | muy_altas)
    return parte

# Primero se cuenta (antes de anular), después se anula
marcas = {n: marcar_valores_absurdos(p, columnas_de_medicion, limite_superior, columnas_de_temperatura)
          for n, p in {"X_train": X_train, "X_val": X_val, "X_test": X_test}.items()}

X_train = anular_valores_absurdos(X_train, columnas_de_medicion, limite_superior, columnas_de_temperatura)
X_val = anular_valores_absurdos(X_val, columnas_de_medicion, limite_superior, columnas_de_temperatura)
X_test = anular_valores_absurdos(X_test, columnas_de_medicion, limite_superior, columnas_de_temperatura)

print("Celdas anuladas por ser absurdas, por parte")
display(tabla_simple(pd.DataFrame({"Celdas anuladas": {n: int(negativas.sum().sum() + altos.sum().sum()) for n, (negativas, altos) in marcas.items()}}), precision=0))

anuladas_por_columna = pd.DataFrame({
    "Negativas (X_train)": marcas["X_train"][0].sum(),
    "Muy altas (X_train)": marcas["X_train"][1].sum(),
    "X_val": marcas["X_val"][0].sum() + marcas["X_val"][1].sum(),
    "X_test": marcas["X_test"][0].sum() + marcas["X_test"][1].sum()})
anuladas_por_columna = anuladas_por_columna[anuladas_por_columna.sum(axis=1) > 0]       # solo las columnas con alguna celda anulada
anuladas_por_columna = anuladas_por_columna.assign(Total=anuladas_por_columna.sum(axis=1)).sort_values("Total", ascending=False)
anuladas_por_columna.insert(0, "Hoja", [c.split(" | ")[0] if " | " in c else "Consolidado EE" for c in anuladas_por_columna.index])

columnas_train = int(((anuladas_por_columna["Negativas (X_train)"] + anuladas_por_columna["Muy altas (X_train)"]) > 0).sum())
print(f"Columnas con celdas anuladas: {len(anuladas_por_columna)} de {len(columnas_de_medicion)} en alguna de las tres partes "
      f"({columnas_train} en X_train). Se muestran las 15 con más celdas anuladas.")
display(tabla_simple(anuladas_por_columna.head(15).rename_axis("Columna").reset_index(), precision=0))


Celdas anuladas por ser absurdas, por parte


,Celdas anuladas
X_train,389
X_val,46
X_test,30


Columnas con celdas anuladas: 96 de 262 en alguna de las tres partes (91 en X_train). Se muestran las 15 con más celdas anuladas.


,Columna,Hoja,Negativas (X_train),Muy altas (X_train),X_val,X_test,Total
0,Resto Serv (Kw),Consolidado EE,25,2,8,1,36
1,Consolidado KPI | EE Resto Serv / Hl,Consolidado KPI,23,4,8,1,36
2,Consolidado KPI | Aire Servicios / Hl,Consolidado KPI,25,3,0,0,28
3,Consolidado Aire | Aire Servicios (M3),Consolidado Aire,25,0,0,0,25
4,Restos Planta (Kw),Consolidado EE,14,2,1,0,17
5,Consolidado KPI | EE Resto Planta / Hl,Consolidado KPI,12,3,1,0,16
6,Consolidado KPI | EE Agua / Hl,Consolidado KPI,5,4,0,0,9
7,Consolidado KPI | Aire L4 / Hl,Consolidado KPI,0,3,3,3,9
8,Consolidado KPI | ET Linea 4/Hl,Consolidado KPI,0,2,4,3,9
9,Consolidado KPI | EE Linea 4 / Hl,Consolidado KPI,0,3,4,1,8


### Paso 4: valores faltantes que quedan

Después de descartar columnas (paso 1), anular los días inválidos (paso 2) y los valores absurdos (paso 3), se mide cuántos vacíos quedan, por hoja y por columna, en cada parte. Es solo un diagnóstico: **este paso no modifica la tabla**.

**Qué se encontró (medido en `X_train`):**
- Faltan 2,4 % de las celdas de medición. 234 de las 262 columnas tienen 5 % o menos de vacíos; 21 tienen entre 5 % y 20 %; 7 superan 20 % (los silos de bagazo, los cocimientos diarios y las razones por hectolitro de la Línea 5).
- Por hoja, las que más vacíos tienen son Totalizadores Glicol (8,6 %), Consolidado KPI (6,9 %) y Totalizadores Producción (6,3 %).
- La mayoría de los vacíos son cortos: 204 columnas nunca tienen más de 3 días seguidos sin dato, y 2.733 de los 5.259 vacíos de entrenamiento están en tramos de 3 días o menos. Solo 5 columnas tienen tramos de más de 30 días sin dato.
- En validación y prueba las razones del KPI de la Línea 5 faltan en 61 % y 52 % de los días (contra 35 % en entrenamiento): esas columnas se están dejando de medir.

**Decisión:** no se rellena ningún vacío en la limpieza. Los vacíos se dejan como `NaN` y su tratamiento se define una sola vez, con la misma regla para todos los modelos, dentro de un `Pipeline` ajustado solo con entrenamiento (cierre de la Fase 2).

**Por qué:**
1. Rellenar con la mediana, el promedio o el valor vecino ahora usaría información de todo el período. La mediana de `X_train` solo puede calcularse dentro de cada tramo de validación temporal, no antes.
2. Los rezagos y promedios móviles de la etapa siguiente se calculan sobre días consecutivos. Un relleno previo les inventaría un dato que nunca existió.
3. Es una porción chica de la tabla (2,4 %), así que no se justifica perder la señal de "acá no hubo dato".
4. Para comparar modelos hace falta que todos reciban el mismo tratamiento (ver ADR 0008).

**Qué queda para la selección:** las columnas cuya disponibilidad cae en validación y prueba (por ejemplo, las razones de la Línea 5) se revisan al elegir variables; una variable que ya no se mide no sirve para predecir el futuro.


In [7]:
def racha_maxima_de_vacios(columna):
    """Mayor cantidad de días seguidos sin dato en una columna."""
    vacios = columna.isna()
    tramo = (vacios != vacios.shift()).cumsum()                 # un número nuevo cada vez que cambia entre dato y vacío
    return int((vacios.groupby(tramo).transform("sum") * vacios).max())   # largo del tramo vacío más largo

faltantes = pd.DataFrame({
    "% faltante X_train": 100 * X_train[columnas_de_medicion].isna().mean(),
    "% faltante X_val": 100 * X_val[columnas_de_medicion].isna().mean(),
    "% faltante X_test": 100 * X_test[columnas_de_medicion].isna().mean(),
    "Racha máxima X_train (días)": X_train[columnas_de_medicion].apply(racha_maxima_de_vacios)})
faltantes.insert(0, "Hoja", [c.split(" | ")[0] if " | " in c else "Consolidado EE" for c in faltantes.index])

celdas = X_train[columnas_de_medicion].size
vacias = int(X_train[columnas_de_medicion].isna().sum().sum())
print(f"X_train: {vacias:,} de {celdas:,} celdas de medición están vacías ({100 * vacias / celdas:.1f} %)".replace(",", "."))

rachas = faltantes["Racha máxima X_train (días)"]
print(f"Columnas con tramos vacíos de 3 días o menos: {int((rachas <= 3).sum())}; con tramos de más de 30 días: {int((rachas > 30).sum())}")

print("Faltantes por hoja: promedio del % de vacíos de sus columnas, en cada parte")
display(tabla_simple(faltantes.groupby("Hoja")[["% faltante X_train", "% faltante X_val", "% faltante X_test"]].mean()
                     .sort_values("% faltante X_train", ascending=False), precision=1))

print("Las 10 columnas con más faltantes en X_train")
display(tabla_simple(faltantes.sort_values("% faltante X_train", ascending=False).head(10).rename_axis("Columna").reset_index(), precision=1))

print("Columnas con más de 40 % de faltantes en validación o prueba (disponibilidad que cae)")
cayeron = faltantes[(faltantes["% faltante X_val"] > 40) | (faltantes["% faltante X_test"] > 40)]
display(tabla_simple(cayeron.rename_axis("Columna").reset_index(), precision=1))

X_train: 5.259 de 222.176 celdas de medición están vacías (2.4 %)
Columnas con tramos vacíos de 3 días o menos: 204; con tramos de más de 30 días: 5
Faltantes por hoja: promedio del % de vacíos de sus columnas, en cada parte


,% faltante X_train,% faltante X_val,% faltante X_test
Hoja,,,
Totalizadores Glicol,8.6,1.6,0.5
Consolidado KPI,6.9,17.8,13.1
Totalizadores Produccion,6.3,1.6,0.6
Consolidado Produccion,3.3,1.6,0.5
Consolidado EE,2.5,3.2,1.7
Totalizadores Efluentes,0.7,1.6,0.5
Consolidado Aire,0.5,1.6,0.5
Totalizadores Gas y Vapor,0.3,1.7,0.5
Totalizadores Agua,0.3,1.7,0.6


Las 10 columnas con más faltantes en X_train


,Columna,Hoja,% faltante X_train,% faltante X_val,% faltante X_test,Racha máxima X_train (días)
0,Totalizadores Produccion | Nivel Silo Bagazo Norte (1),Totalizadores Produccion,36.6,1.6,1.6,308
1,Totalizadores Produccion | Nivel Silo Bagazo Sur (2),Totalizadores Produccion,36.6,1.6,0.5,308
2,Consolidado KPI | Aire L5 / Hl,Consolidado KPI,34.6,61.0,51.6,22
3,Consolidado KPI | Agua Linea 5/Hl,Consolidado KPI,34.6,61.0,51.6,22
4,Consolidado KPI | ET Linea 5/Hl,Consolidado KPI,34.6,61.0,51.6,22
5,Consolidado Produccion | Cocimientos Diarios,Consolidado Produccion,30.5,1.6,0.5,257
6,Totalizadores Produccion | Cocimientos Diarios,Totalizadores Produccion,30.5,1.6,0.5,257
7,Consolidado KPI | EE Linea 3 / Hl,Consolidado KPI,14.6,34.1,25.8,23
8,Consolidado KPI | ET Linea 3/Hl,Consolidado KPI,13.2,33.5,26.4,23
9,Consolidado KPI | Agua Linea 3/Hl,Consolidado KPI,13.1,33.5,25.3,23


Columnas con más de 40 % de faltantes en validación o prueba (disponibilidad que cae)


,Columna,Hoja,% faltante X_train,% faltante X_val,% faltante X_test,Racha máxima X_train (días)
0,Consolidado KPI | Agua Linea 5/Hl,Consolidado KPI,34.6,61.0,51.6,22
1,Consolidado KPI | ET Linea 5/Hl,Consolidado KPI,34.6,61.0,51.6,22
2,Consolidado KPI | Aire L5 / Hl,Consolidado KPI,34.6,61.0,51.6,22


### Paso 5: empaquetamos la limpieza como función reutilizable

Los pasos 1 a 3 quedaron programados en `src/limpieza.py` para que `predict.py` aplique la misma limpieza a datos nuevos sin reescribirla:

- `aprender_reglas_de_limpieza(X_train)` mide, **solo con entrenamiento**, qué columnas se conservan, cuáles son de energía y cuáles son los límites del percentil 99.
- `limpiar(tabla, reglas)` aplica esas reglas a cualquier tabla, sin recalcular nada. No borra filas ni rellena vacíos.

La celda siguiente parte de las tablas sin limpiar y comprueba que las funciones dan **exactamente** las mismas tres tablas que los pasos manuales de arriba. Las pruebas de las funciones están en `tests/test_limpieza.py`.

In [8]:
from limpieza import aprender_reglas_de_limpieza, limpiar

# Se vuelve a dividir la tabla completa para partir de las tablas sin limpiar
sin_limpiar_train, sin_limpiar_val, sin_limpiar_test = dividir_por_fecha(tabla_completa)

reglas = aprender_reglas_de_limpieza(sin_limpiar_train)          # solo con entrenamiento
X_train_funcion = limpiar(sin_limpiar_train, reglas)
X_val_funcion = limpiar(sin_limpiar_val, reglas)
X_test_funcion = limpiar(sin_limpiar_test, reglas)

print("True si las funciones dan el mismo resultado que el procedimiento manual")
display(tabla_simple(pd.DataFrame({"Igual": {
    "X_train": X_train.equals(X_train_funcion),
    "X_val": X_val.equals(X_val_funcion),
    "X_test": X_test.equals(X_test_funcion)}}), precision=0))

True si las funciones dan el mismo resultado que el procedimiento manual


,Igual
X_train,True
X_val,True
X_test,True


## 4. Asimetría: se mide ahora, se transforma después

Aquí solo se **mide** qué tan asimétricas son las columnas de medición en `X_train` (asimetría de Fisher-Pearson: 0 es simétrica, y cuanto más lejos de 0, más larga es la cola de un lado). **No se transforma nada en esta sección.**

**Decisión:** la transformación para corregir la asimetría (Yeo-Johnson) y el escalado se aplican **después de crear las variables nuevas**, dentro de un `Pipeline` de scikit-learn ajustado solo con entrenamiento, y solo para los modelos que lo necesitan (Ridge y Lasso; los modelos de árboles no). El objetivo no se transforma: se queda en kW porque el error (MAE) se mide en kW. Se usa Yeo-Johnson y no Box-Cox porque Box-Cox exige valores estrictamente positivos y varias columnas tienen ceros. Detalle en el ADR 0010.

**Por qué no antes**, aunque en el orden clásico (limpieza, transformación, escalado y luego ingeniería de características) la transformación viene antes. Lo que hay que resolver antes de crear variables ya está resuelto (tipos de dato, duplicados, valores faltantes y valores absurdos), porque esas operaciones son las que las variables nuevas necesitan limpias. La transformación no, y hacerla antes tiene costos concretos:
1. **Los valores absurdos sí van antes:** un `Sala Maq (Kw)` de 4.208.132 dentro de un promedio móvil de 7 días arruinaría la variable durante una semana. La asimetría no tiene ese efecto.
2. **Una razón cambia de significado:** el frío sobre el total de la planta con kW reales es el porcentaje que consume el frío. Con columnas ya estandarizadas, el denominador pasa por valores cercanos a 0 y negativos, y la razón explota o cambia de signo.
3. **El promedio de valores transformados no es el valor transformado del promedio:** con tres días de 1.000, 1.000 y 100.000 kW, el promedio real es 34.000 y el promedio de los logaritmos equivale a unos 4.400. Para energía interesa lo consumido, es decir el promedio real. (El rezago sí conmuta con la transformación, así que ahí el orden da lo mismo.)
4. **Las variables nuevas traen su propia asimetría** (las razones por hectolitro ya llegan a 14 en esta tabla), así que habría que transformar de nuevo después y la transformación tendría que ir al final de todos modos.
5. **Los parámetros se aprenden:** Yeo-Johnson y el escalador estiman valores con los datos. Ajustarlos una sola vez con todo `X_train` antes de validar haría que cada tramo de validación ya haya influido; dentro del `Pipeline` se reajustan en cada tramo de `TimeSeriesSplit`, como pide la consigna 2.4.

### Cómo leer las tablas de esta sección

Las tablas se calculan solo con `X_train` (ya limpio) y miden la **asimetría** de cada columna de medición. La asimetría dice hacia qué lado se estira la distribución de una columna y **puede ser positiva o negativa**:

- **Cerca de 0:** la columna es simétrica (como una campana).
- **Positiva y grande** (por ejemplo +5): casi todos los días tienen valores chicos y unos pocos días tienen valores enormes (cola larga hacia la derecha). Es el caso típico de un consumo que casi siempre es bajo y a veces se dispara.
- **Negativa y grande** (por ejemplo −4): casi todos los días tienen valores altos y unos pocos días tienen valores muy bajos (cola larga hacia la izquierda). Por ejemplo, una línea que casi siempre trabaja a pleno y a veces se detiene.

El signo importa para elegir la transformación: el logaritmo solo corrige la asimetría positiva, mientras que Yeo-Johnson corrige las dos. Por eso las tablas separan los dos signos. Se considera "fuerte" una asimetría de más de 1 en valor absoluto.

- **Tabla 1, columnas según su asimetría:** cuenta cuántas de las 262 columnas hay en cada grupo (positiva fuerte, simétrica, negativa fuerte). Después cuenta las que tienen algún valor negativo y las que tienen algún cero, porque eso decide qué transformación se puede usar (Box-Cox no admite ceros ni valores negativos).
- **Tabla 2, asimetría por hoja:** para cada hoja del Excel, cuántas columnas tiene, cuántas tienen asimetría positiva fuerte, cuántas negativa fuerte y cuánto vale la mediana de la asimetría **en valor absoluto** (la columna del medio, un resumen del tamaño de la asimetría de la hoja).
- **Tabla 3, las 10 columnas más asimétricas:** las que más se alejan de lo simétrico, con su **signo**.

Al final se muestra la asimetría de `Frio (Kw)` para comparar.

In [9]:
valores_train = X_train[columnas_de_medicion].apply(pd.to_numeric, errors="coerce")
asimetria = valores_train.skew()                  # asimetría de cada columna (0 = simétrica; puede ser negativa)
tamano = asimetria.abs()                          # cuánta asimetría hay, sin importar el signo

print("Tabla 1. Cuántas columnas hay según su asimetría")
display(tabla_simple(pd.DataFrame({"Columnas": {
    "Asimetría positiva fuerte (más de +1)": int((asimetria > 1).sum()),
    "Casi simétricas (entre -1 y +1)": int((tamano <= 1).sum()),
    "Asimetría negativa fuerte (menos de -1)": int((asimetria < -1).sum()),
    "Con algún valor negativo (Box-Cox no sirve)": int((valores_train.min() < 0).sum()),
    "Con algún cero (Box-Cox no sirve)": int((valores_train == 0).any().sum())}}), precision=0))

hoja_de_columna = pd.Series([c.split(" | ")[0] if " | " in c else "Consolidado EE" for c in valores_train.columns],
                            index=valores_train.columns)
por_hoja = pd.DataFrame({"Columnas": hoja_de_columna.value_counts(),
                         "Positiva fuerte (> +1)": (asimetria > 1).groupby(hoja_de_columna).sum(),
                         "Negativa fuerte (< -1)": (asimetria < -1).groupby(hoja_de_columna).sum(),
                         "Mediana del tamaño (valor absoluto)": tamano.groupby(hoja_de_columna).median()})
print("Tabla 2. Asimetría de las columnas de cada hoja (de más a menos asimétrica)")
display(tabla_simple(por_hoja.sort_values("Mediana del tamaño (valor absoluto)", ascending=False), precision=1))

mas_asimetricas = asimetria.reindex(tamano.sort_values(ascending=False).head(10).index)     # se ordena por tamaño, se muestra con signo
print("Tabla 3. Las 10 columnas más asimétricas (con su signo)")
display(tabla_simple(mas_asimetricas.rename("Asimetría").rename_axis("Columna").reset_index(), precision=1))
print(f"Para comparar: la asimetría de Frio (Kw) es {asimetria['Frio (Kw)']:.2f} (casi simétrica) y es el objetivo, que no se transforma.")

Tabla 1. Cuántas columnas hay según su asimetría


,Columnas
Asimetría positiva fuerte (más de +1),113
Casi simétricas (entre -1 y +1),124
Asimetría negativa fuerte (menos de -1),25
Con algún valor negativo (Box-Cox no sirve),1
Con algún cero (Box-Cox no sirve),173


Tabla 2. Asimetría de las columnas de cada hoja (de más a menos asimétrica)


,Columnas,Positiva fuerte (> +1),Negativa fuerte (< -1),Mediana del tamaño (valor absoluto)
Totalizadores Efluentes,5,5,0,12.4
Consolidado KPI,54,51,1,6.8
Totalizadores Produccion,17,8,1,1.4
Totalizadores Agua,37,18,2,1.2
Totalizadores Glicol,1,1,0,1.1
Totalizadores Gas y Vapor,14,6,1,0.9
Totalizadores Energia,49,12,7,0.7
Consolidado Produccion,10,1,2,0.7
Consolidado Agua,16,4,2,0.7
Totalizadores Aire,7,0,2,0.7


Tabla 3. Las 10 columnas más asimétricas (con su signo)


,Columna,Asimetría
0,Totalizadores Efluentes | Totalizador Bba P2,15.9
1,Consolidado KPI | EE Agua / Hl,14.4
2,Consolidado KPI | EE Aire / Hl,14.1
3,Consolidado KPI | EE Resto Planta / Hl,14.1
4,Consolidado KPI | Aire Planta / Hl,14.1
5,Consolidado KPI | EE Sala Maq / Hl,13.5
6,Consolidado KPI | Aire Bodega / Hl,13.4
7,Consolidado KPI | EE Servicios / Hl,13.4
8,Totalizadores Efluentes | Totalizador Bba P4,13.4
9,Consolidado KPI | Aire Cocina / Hl,13.2


Para comparar: la asimetría de Frio (Kw) es -0.31 (casi simétrica) y es el objetivo, que no se transforma.


## 5. Columnas Meta: Decidimos ignorar la sugerencia del TP

La consigna 2.2 sugiere crear una variable de eficiencia energética (consumo real contra meta) con las 51 columnas `Meta` de Consolidado KPI. Es una sugerencia y no una exigencia, y decidimos no usarla. Todas las columnas `Meta` ya quedaron fuera por el criterio de más de 40 % de faltantes (ADR 0006); esta sección justifica que no conviene hacerles una excepción. Las cifras se miden solo con `X_train`.

**Por qué no:**
1. **Son faltantes MAR:** las metas empiezan a registrarse el 01/07/2021; antes no existen (100 % de vacíos en el primer año). Solo se podrían usar rellenando hacia atrás, que sería inventar un año de metas, o descartando el primer año, que son 367 de los 848 días de entrenamiento.
2. **Casi no varían:** dentro de un mismo mes hay un único valor, y a lo largo de todo el período cada columna toma pocos valores distintos (ver tabla). No aportan variación día a día; lo que cambia de mes a mes ya lo capta la variable mes del calendario.

### Cómo leer las tablas de esta sección

Las tablas se calculan solo con `X_train` (sin limpiar) y muestran cómo se comportan las 51 columnas `Meta`. Una columna `Meta` es un valor de referencia fijado por la planta (por ejemplo, el consumo de energía por hectolitro que se espera para toda la planta).

- **Tabla 1, ejemplo de una columna mes a mes:** cada fila es un mes. *Valor de la meta ese mes* es el número que tuvo la columna `Meta EE Planta` durante todo ese mes. *Valores distintos dentro del mes* cuenta cuántos números diferentes tuvo en ese mes: si siempre da 1, la meta **no cambia dentro del mes**, solo de un mes al siguiente.
- **Tabla 2, resumen de las 51 columnas:** las mismas características se calculan para cada una de las 51 columnas y se muestran tres casos: la columna con el valor más bajo, la típica (la mediana) y la que tiene el valor más alto. Las filas son:
  - *Valores distintos que toma:* cuántos números diferentes aparecen en los días con meta. Si da 12, la columna toma 12 valores, más o menos uno por mes.
  - *% de días con el valor más frecuente:* en qué porcentaje de los días la columna vale lo mismo que su valor más repetido. Cerca de 13 % significa que ningún valor domina; 100 % significa que la columna es constante.
  - *Veces que cambia de un día al siguiente:* cuántas veces el valor de un día es distinto del día anterior. Si da 11, cambia una vez por mes.
  - *Variación relativa:* qué tan lejos se mueve la columna de su promedio (el desvío dividido por el promedio). 0,11 significa que varía alrededor de un 11 %.
  - *% de días sin meta en X_train:* porcentaje de días de entrenamiento en que la columna no tiene dato.
- **Tabla 3, días sin meta en cada parte:** el porcentaje de días sin meta en `X_train`, `X_val` y `X_test`. Muestra que faltan en entrenamiento porque las metas empiezan en julio de 2021, y que después casi no faltan.

In [10]:
columnas_meta = [c for c in sin_limpiar_train.columns if c.startswith("Consolidado KPI | Meta")]
metas = sin_limpiar_train[columnas_meta].apply(pd.to_numeric, errors="coerce")
con_meta = metas.dropna(how="all")                  # solo los días en que las metas ya existen
primer_dia = sin_limpiar_train.loc[con_meta.index[0], "DIA"].strftime("%d/%m/%Y")
mes_del_dia = sin_limpiar_train.loc[con_meta.index, "DIA"].dt.to_period("M")

print(f"Las {len(columnas_meta)} columnas Meta empiezan el {primer_dia}: {len(con_meta)} de los {len(sin_limpiar_train)} días de X_train tienen metas.")

ejemplo = "Consolidado KPI | Meta EE Planta"
print(f"Tabla 1. Valor de «{ejemplo}» en cada mes")
por_mes = con_meta[ejemplo].groupby(mes_del_dia.values).agg(["first", "nunique"])
por_mes.columns = ["Valor de la meta ese mes", "Valores distintos dentro del mes"]
por_mes.index = por_mes.index.astype(str).rename("Mes")
display(tabla_simple(por_mes.reset_index(), precision=2))

variacion = pd.DataFrame({
    "Valores distintos que toma": con_meta.nunique(),
    "% de días con el valor más frecuente": con_meta.apply(lambda s: 100 * s.value_counts(normalize=True).iloc[0]),
    "Veces que cambia de un día al siguiente": (con_meta.diff().abs() > 0).sum(),
    "Variación relativa (desvío / promedio)": con_meta.std() / con_meta.mean().abs(),
    "% de días sin meta en X_train": 100 * metas.isna().mean()})
resumen = variacion.agg(["min", "median", "max"]).T.rename(columns={
    "min": "Columna con el valor más bajo", "median": "Columna típica (mediana)", "max": "Columna con el valor más alto"})
print(f"Tabla 2. Resumen de las {len(columnas_meta)} columnas Meta")
display(tabla_simple(resumen.rename_axis("Característica").reset_index(), precision=2))

disponibilidad = pd.DataFrame({"% de días sin meta": {
    "X_train": 100 * metas.isna().mean().mean(),
    "X_val": 100 * sin_limpiar_val[columnas_meta].apply(pd.to_numeric, errors="coerce").isna().mean().mean(),
    "X_test": 100 * sin_limpiar_test[columnas_meta].apply(pd.to_numeric, errors="coerce").isna().mean().mean()}})
print("Tabla 3. Días sin meta en cada parte")
display(tabla_simple(disponibilidad.rename_axis("Parte").reset_index(), precision=1))

Las 51 columnas Meta empiezan el 01/07/2021: 481 de los 848 días de X_train tienen metas.
Tabla 1. Valor de «Consolidado KPI | Meta EE Planta» en cada mes


,Mes,Valor de la meta ese mes,Valores distintos dentro del mes
0,2021-07,6.90,1
1,2021-08,6.80,1
2,2021-09,6.80,1
3,2021-10,6.70,1
4,2021-11,6.60,1
5,2021-12,7.80,1
6,2022-01,7.80,1
7,2022-02,6.50,1
8,2022-03,6.30,1
9,2022-04,6.30,1


Tabla 2. Resumen de las 51 columnas Meta


,Característica,Columna con el valor más bajo,Columna típica (mediana),Columna con el valor más alto
0,Valores distintos que toma,1.00,12.00,12.00
1,% de días con el valor más frecuente,12.68,12.68,100.00
2,Veces que cambia de un día al siguiente,0.00,11.00,13.00
3,Variación relativa (desvío / promedio),0.00,0.11,0.13
4,% de días sin meta en X_train,43.28,43.28,43.28


Tabla 3. Días sin meta en cada parte


,Parte,% de días sin meta
0,X_train,43.3
1,X_val,1.6
2,X_test,0.5


## 6. Temperatura (Open-Meteo) (por sugerencia del TP)

La consigna recomienda Open-Meteo para obtener la temperatura ambiente de Tecate. Antes de decidir si vale la pena o no usar esta información se hace una **prueba exploratoria, solo con `X_train`**, para ver si la temperatura aporta algo que no aporten ya los rezagos del frío y el calendario.

### Cómo leer las tablas de esta sección

Todas las tablas usan solo los días válidos de `X_train` y la temperatura media diaria de Tecate (la unión es por fecha).

- **Tabla 1, correlación según el desfase en días:** la correlación de Spearman entre `Frio (Kw)` de un día y la temperatura de ese mismo día (desfase 0) o de días anteriores y posteriores (desfase −3 a +3). Sirve para ver si las fechas de los Excel están alineadas con el calendario: si lo estuvieran mal, el valor más alto no aparecería en el desfase 0. Como la estación domina las dos series, es esperable que los valores sean parecidos entre sí.
- **Tabla 2, correlación total y quitando la estación:** la primera fila es la correlación entre el frío y la temperatura tal cual. La segunda le resta a cada serie el promedio de su mes y correlaciona lo que queda, es decir, si **dentro de un mismo mes** los días más calientes consumen más frío. Un valor cerca de 0 significa que la temperatura solo se relaciona con el frío a través de la estación.
- **Tabla 3, error de predicción con y sin temperatura:** para predecir el frío del día siguiente se entrenan dos modelos simples con tres grupos de variables (rezagos del frío y estacionalidad; más la temperatura de hoy; más la temperatura real de mañana). El error (MAE, en kW) se mide en 5 tramos de `TimeSeriesSplit` dentro de `X_train` y se muestra el promedio. Menor es mejor. Lo que importa es la **diferencia entre filas**, no el valor absoluto, porque son modelos rápidos con pocos datos por tramo.

In [11]:
import json
import urllib.parse
import urllib.request

import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit

from auxiliar_functions import SEMILLA

archivo_clima = Path("../data/interim/clima_tecate_prueba.json")
if not archivo_clima.exists():
    consulta = {"latitude": 32.5734, "longitude": -116.6254,         # Tecate, Baja California (aproximado)
                "start_date": X_train["DIA"].min().strftime("%Y-%m-%d"), "end_date": X_val["DIA"].max().strftime("%Y-%m-%d"),
                "daily": "temperature_2m_mean", "timezone": "America/Tijuana"}
    try:
        with urllib.request.urlopen("https://archive-api.open-meteo.com/v1/archive?" + urllib.parse.urlencode(consulta), timeout=30) as respuesta:
            archivo_clima.parent.mkdir(exist_ok=True)
            archivo_clima.write_text(respuesta.read().decode("utf8"), encoding="utf8")
    except Exception as error:
        print(f"No se pudo descargar la temperatura ({type(error).__name__}); no se puede repetir esta prueba sin conexión.")


def evaluar_temperatura(entrenamiento, archivo):
    """Imprime las tres tablas de la prueba de temperatura usando solo el tramo de entrenamiento."""
    clima = pd.DataFrame(json.loads(archivo.read_text(encoding="utf8"))["daily"])
    clima["DIA"] = pd.to_datetime(clima["time"])
    t = entrenamiento[["DIA", "Frio (Kw)", "objetivo", "valido"]].merge(
        clima[["DIA", "temperature_2m_mean"]].rename(columns={"temperature_2m_mean": "temp"}), on="DIA", how="left")
    frio = t["Frio (Kw)"]

    # Tabla 1: la temperatura de t+k contra el frío de t
    desfases = pd.DataFrame({"Spearman con el frío": {
        f"{k:+d} días": frio.corr(t["temp"].shift(-k), method="spearman") for k in range(-3, 4)}})
    print("Tabla 1. Correlación entre el frío y la temperatura según el desfase")
    display(tabla_simple(desfases.rename_axis("Desfase de la temperatura").reset_index(), precision=3))

    # Tabla 2: total y sin la estación (se resta el promedio de cada mes)
    validos = t[t["valido"]].copy()
    mes = validos["DIA"].dt.month
    desvio_frio = validos["Frio (Kw)"] - validos.groupby(mes)["Frio (Kw)"].transform("mean")
    desvio_temp = validos["temp"] - validos.groupby(mes)["temp"].transform("mean")
    print("Tabla 2. Correlación entre el frío y la temperatura, con y sin la estación")
    display(tabla_simple(pd.DataFrame({"Spearman": {
        "Frío y temperatura, tal cual": validos["Frio (Kw)"].corr(validos["temp"], method="spearman"),
        "Desvíos respecto del promedio de su mes": desvio_frio.corr(desvio_temp, method="spearman")}}).rename_axis("Comparación").reset_index(), precision=3))

    # Tabla 3: error al predecir el frío de mañana
    dia_del_anio = t["DIA"].dt.dayofyear
    datos = pd.DataFrame({
        "frio_hoy": frio, "frio_ayer": frio.shift(1), "media_7_dias": frio.rolling(7, min_periods=5).mean(),
        "seno": np.sin(2 * np.pi * dia_del_anio / 365.25), "coseno": np.cos(2 * np.pi * dia_del_anio / 365.25),
        "temp_hoy": t["temp"], "temp_manana": t["temp"].shift(-1), "objetivo": t["objetivo"]}).dropna()
    grupos = {"Sin temperatura": ["frio_hoy", "frio_ayer", "media_7_dias", "seno", "coseno"],
              "Con la temperatura de hoy": ["frio_hoy", "frio_ayer", "media_7_dias", "seno", "coseno", "temp_hoy"],
              "Con la temperatura de hoy y la real de mañana": ["frio_hoy", "frio_ayer", "media_7_dias", "seno", "coseno", "temp_hoy", "temp_manana"]}
    modelos = {"Ridge": lambda: Ridge(alpha=1.0),
               "Gradient boosting": lambda: HistGradientBoostingRegressor(random_state=SEMILLA, max_iter=150, learning_rate=0.05)}
    y = datos["objetivo"].to_numpy()
    maes = {}
    for nombre_grupo, columnas in grupos.items():
        X = datos[columnas].to_numpy()
        maes[nombre_grupo] = {}
        for nombre_modelo, crear in modelos.items():
            errores = [np.mean(np.abs(crear().fit(X[ent], y[ent]).predict(X[val]) - y[val]))
                       for ent, val in TimeSeriesSplit(5).split(X)]
            maes[nombre_grupo][nombre_modelo] = np.mean(errores)
    print(f"Tabla 3. Error medio (MAE, en kW) al predecir el frío de mañana, con {len(datos)} días de X_train")
    display(tabla_simple(pd.DataFrame(maes).T.rename_axis("Variables usadas").reset_index(), precision=0))
    print(f"Para comparar: la persistencia (repetir el frío de hoy) tiene un MAE de {np.mean(np.abs(datos['frio_hoy'] - y)):.0f} kW sobre los mismos días.")


if archivo_clima.exists():
    evaluar_temperatura(X_train, archivo_clima)

Tabla 1. Correlación entre el frío y la temperatura según el desfase


,Desfase de la temperatura,Spearman con el frío
0,-3 días,0.517
1,-2 días,0.523
2,-1 días,0.528
3,+0 días,0.525
4,+1 días,0.533
5,+2 días,0.541
6,+3 días,0.548


Tabla 2. Correlación entre el frío y la temperatura, con y sin la estación


,Comparación,Spearman
0,"Frío y temperatura, tal cual",0.525
1,Desvíos respecto del promedio de su mes,-0.014


Tabla 3. Error medio (MAE, en kW) al predecir el frío de mañana, con 798 días de X_train


,Variables usadas,Ridge,Gradient boosting
0,Sin temperatura,3215,4382
1,Con la temperatura de hoy,3179,4216
2,Con la temperatura de hoy y la real de mañana,3138,4225


Para comparar: la persistencia (repetir el frío de hoy) tiene un MAE de 3189 kW sobre los mismos días.


**Decisión: no se incorpora la temperatura** (ADR 0011). Esta sección es solo la evidencia. **No forma parte del pipeline**: la celda descarga una vez los datos públicos de Open-Meteo (solo coordenadas y fechas) y los guarda en `data/interim/`, carpeta que no se versiona. Si no hay conexión y no hay copia guardada, la celda avisa y el resto del notebook sigue funcionando.

## 7. Selección de variables, elección del Pipeline y dataset final

Hasta acá se limpió la tabla y se crearon las variables nuevas: el resultado son 337 columnas por día. Esta última sección junta ese resultado con lo que hicieron las otras tareas de la fase. Muestra qué variables quedaron seleccionadas y con qué criterio, compara las variantes del `Pipeline` de scikit-learn con esas variables, explica qué se hace con las dos filas cuyo objetivo cae en el tramo siguiente y guarda el dataset final. Las decisiones se toman solo con el tramo de entrenamiento; la validación se usa para informar, y la prueba no se mira más que para comprobar que la transformación aprendida con entrenamiento se puede aplicar. En las celdas no se aprende nada nuevo: se llama a funciones de `src/` que ya tienen sus pruebas.

Para partir de lo mismo que usa el script `src/preprocessing_pipeline.py`, se vuelve a correr el preprocesamiento completo (si ya existe la copia de trabajo de la tabla, tarda pocos segundos) y se le pide la lista de variables a `seleccionar_variables`. Esa función no aprende nada en este paso: lee la lista que quedó congelada en `results/seleccion/` y comprueba su huella, porque la selección se hizo y se documentó antes (ADR 0015).

In [12]:
import json

import numpy as np

from auxiliar_functions import huella
from pipeline_sklearn import (ALPHA_ELEGIDO, VARIANTE_POR_DEFECTO, aprender_reglas_de_grupos, armar_pipeline,
                              comparar_variantes, elegir_variante, evaluar_en_validacion, resumir_comparacion,
                              separar_xy)
from preprocessing_pipeline import (_quitar_etiquetas_de_frontera, completar_fase2, dividir_por_fecha,
                                    ejecutar_preprocesamiento)
from seleccion_definitiva import cargar_seleccion_definitiva
from seleccion_de_variables import seleccionar_variables

RAIZ = Path("..").resolve()                                   # el notebook vive en notebooks/
preproceso = ejecutar_preprocesamiento(RAIZ / "data")         # de los Excel a la tabla con las 337 columnas
tabla_final = preproceso["tabla_con_variables"]
entrenamiento, validacion = preproceso["entrenamiento"], preproceso["validacion"]
columnas = seleccionar_variables(entrenamiento, entrenamiento["objetivo"])   # lista congelada: no aprende

def titulo_y_tabla(numero, titulo, tabla, precision=2):
    """Imprime el título de la tabla y la muestra con el estilo del notebook."""
    print(f"Tabla 7.{numero}. {titulo}")
    display(tabla_simple(tabla, precision=precision))

print(f"La tabla tiene {tabla_final.shape[0]} días y {tabla_final.shape[1]} columnas; se eligieron {len(columnas)} variables.")

La tabla tiene 1212 días y 337 columnas; se eligieron 14 variables.


La selección usó un método de filtro, no un wrapper. Un filtro ordena las variables con una medida estadística calculada variable por variable, sin entrenar un modelo para cada subconjunto; un wrapper, como la selección hacia adelante (forward), prueba conjuntos de variables entrenando un modelo cada vez. Acá la medida fue el valor absoluto de la correlación de Spearman con el objetivo, que mide cuánto sube o baja el consumo de mañana junto con cada variable aunque la relación no sea una recta. Se tomaron las primeras variables del ranking y, por el principio de jerarquía del ADR 0014, a cada interacción elegida se le sumaron sus dos variables componentes cuando todavía no estaban. La tabla 7.1 muestra las 14 variables resultantes, cómo entró cada una, su correlación con el objetivo calculada ahora con todo el entrenamiento y el porcentaje de días en que están vacías.

In [13]:
ruta_selecciones = RAIZ / "results/seleccion/ejecucion-20261113T184058Z-969d4545/selecciones_por_metodo.json"
directas = json.loads(ruta_selecciones.read_text(encoding="utf-8"))["spearman"]["seleccion"]["directas"]
etiquetadas = entrenamiento.loc[entrenamiento["objetivo"].notna()]

variables_elegidas = pd.DataFrame({
    "Cómo entró": ["Ranking de Spearman" if c in directas else "Jerarquía (componente de una interacción)"
                   for c in columnas],
    "|Spearman| con el objetivo": etiquetadas[columnas].corrwith(etiquetadas["objetivo"], method="spearman").abs(),
    "Vacíos en entrenamiento (%)": entrenamiento[columnas].isna().mean() * 100,
}, index=pd.Index(columnas, name="Variable"))
titulo_y_tabla(1, "Las 14 variables seleccionadas, en el orden en que quedaron congeladas", variables_elegidas)

Tabla 7.1. Las 14 variables seleccionadas, en el orden en que quedaron congeladas


,Cómo entró,|Spearman| con el objetivo,Vacíos en entrenamiento (%)
Variable,,,
Frio (Kw),Ranking de Spearman,0.85,2.00
Frio (Kw) x frio_hace_2_dias,Ranking de Spearman,0.84,4.25
frio_media_7_dias,Ranking de Spearman,0.83,0.24
Sala Maq (Kw),Ranking de Spearman,0.83,2.36
Frio (Kw) x Consolidado Agua | Agua Planta (Hl),Ranking de Spearman,0.83,2.00
Servicios (Kw),Ranking de Spearman,0.82,2.00
Frio (Kw) x Totalizadores Energia | KW Trafo 9,Ranking de Spearman,0.81,2.00
dia_anio_coseno,Ranking de Spearman,0.80,0.00
frio_hace_2_dias x Consolidado Agua | Agua Planta (Hl),Ranking de Spearman,0.80,2.48


Spearman no fue el método con menor error en validación: el menor MAE lo tuvo Pearson, pero la diferencia es más chica que el 1 % que se había fijado como margen de equivalencia y Spearman necesita menos variables, así que se lo eligió por simplicidad. La tabla 7.2 deja los cinco métodos lado a lado, y las últimas dos columnas miden el error sobre las filas donde también existen los baselines. Hay que leerla con una advertencia: la validación se usó para elegir entre los cinco, de modo que el error de Spearman ahí es algo optimista y no es el rendimiento final del modelo.

In [14]:
definitiva = cargar_seleccion_definitiva()                   # verifica la huella del archivo congelado
comparacion = pd.DataFrame(definitiva["comparacion"]).set_index("metodo").rename(columns={
    "variables": "Variables", "columnas_efectivas": "Columnas efectivas", "mae": "MAE en validación (kW)",
    "filas": "Filas", "mae_en_filas_baseline": "MAE en las filas con baselines (kW)"})
comparacion.index.name = "Método"
titulo_y_tabla(2, "Los cinco métodos de selección comparados en validación (mismo Ridge, mismas filas)",
               comparacion.sort_values("MAE en validación (kW)"))

Tabla 7.2. Los cinco métodos de selección comparados en validación (mismo Ridge, mismas filas)


,MAE en validación (kW),Variables,Columnas efectivas,Filas,MAE en las filas con baselines (kW)
Método,,,,,
pearson,2239.58,19,37,176,2171.97
spearman,2247.82,14,27,176,2173.40
forward,2377.22,30,59,176,2283.36
lasso,2723.35,268,532,176,2532.07
mutua,2736.27,311,619,176,2538.29


Con las 14 variables se repite la comparación de las variantes del `Pipeline` que ya se había hecho con un conjunto provisional de 10. Ridge actúa como juez que mide el error, no como modelo final, que se elige en la fase siguiente. Cada variante se prueba con cuatro valores de la regularización (`alpha`) en cinco pliegues cronológicos del entrenamiento, y de cada una se queda el `alpha` de menor error promedio. Para elegir se usa la regla de un desvío: se calcula el mejor promedio más su desvío entre pliegues y se toma la variante más simple cuyo promedio no pasa de ese límite. La tabla 7.3 muestra el resultado y la variante que quedó dentro del límite con menos pasos.

In [15]:
detalle, referencias = comparar_variantes(entrenamiento, columnas)       # 6 variantes x 4 alphas x 5 pliegues
resumen = resumir_comparacion(detalle)
variante_elegida, limite = elegir_variante(resumen)
alpha_elegido = float(resumen.loc[variante_elegida, "alpha"])

tabla_variantes = resumen.rename(columns={"alpha": "Alpha", "mae_promedio": "MAE promedio (kW)",
                                          "desvio": "Desvío entre pliegues (kW)"})
tabla_variantes["Dentro del límite"] = resumen["mae_promedio"] <= limite
tabla_variantes.index.name = "Variante"
titulo_y_tabla(3, f"Variantes del Pipeline con las 14 variables (límite de un desvío: {limite:,.2f} kW)", tabla_variantes)
print(f"Variante elegida: {variante_elegida} con alpha {alpha_elegido:g}")

Tabla 7.3. Variantes del Pipeline con las 14 variables (límite de un desvío: 3,830.23 kW)


,Alpha,MAE promedio (kW),Desvío entre pliegues (kW),Dentro del límite
Variante,,,,
A,10.00,3381.04,756.06,True
B,100.00,3339.25,517.46,True
C,100.00,3334.15,496.08,True
D,10.00,3341.48,537.41,True
E,100.00,3335.45,481.52,True
F,0.10,3381.31,620.05,True


Variante elegida: A con alpha 10


El error de un mismo método cambia según el tramo donde se mide, y eso le pasa también a los baselines. Cada pliegue cubre otra época del año, con otro nivel de consumo y otra variabilidad, y además cada pliegue entrena con más historia que el anterior. Por eso la tabla 7.4 pone, para cada pliegue, las fechas, el consumo medio y su desvío junto al error de los dos baselines y al de la variante elegida, todos medidos sobre las mismas filas. Lo que importa es comparar dentro de una misma fila de la tabla, nunca un número de un pliegue con el de otro.

In [16]:
mae_elegida = detalle[(detalle["variante"] == variante_elegida) & (detalle["alpha"] == alpha_elegido)].set_index("pliegue")["mae"]
por_pliegue = pd.DataFrame({
    "Filas de ajuste": referencias["filas_ajuste"], "Filas de evaluación": referencias["filas_evaluacion"],
    "Inicio": referencias["inicio"].dt.date.astype(str), "Fin": referencias["fin"].dt.date.astype(str),
    "Consumo medio (kW)": referencias["consumo_medio"], "Desvío del consumo (kW)": referencias["desvio_consumo"],
    "MAE persistencia": referencias["persistencia"], "MAE media móvil": referencias["media_movil"],
    f"MAE variante {variante_elegida}": mae_elegida})
por_pliegue.index.name = "Pliegue"
titulo_y_tabla(4, "Cada pliegue: fechas, consumo y error de los baselines y de la variante elegida", por_pliegue)

Tabla 7.4. Cada pliegue: fechas, consumo y error de los baselines y de la variante elegida


,Filas de ajuste,Filas de evaluación,Inicio,Fin,Consumo medio (kW),Desvío del consumo (kW),MAE persistencia,MAE media móvil,MAE variante A
Pliegue,,,,,,,,,
1,139,138,2020-11-22,2021-04-06,18322.14,6341.99,2576.87,2858.93,4637.29
2,277,131,2021-04-07,2021-09-01,32349.17,4334.97,3478.95,3301.24,3061.21
3,415,137,2021-09-02,2022-01-18,19340.39,8180.92,3240.04,3774.99,3158.76
4,553,134,2022-01-19,2022-06-09,25234.47,6675.43,2976.93,3143.01,2635.63
5,691,137,2022-06-10,2022-10-27,26195.53,6344.51,3974.88,3805.95,3412.30


La tabla 7.5 resume lo anterior y le suma la validación, que se usa solo para informar y no para elegir nada. En los pliegues se promedian los cinco errores; en validación hay un único bloque de fechas, y los baselines se calculan sobre entrenamiento y validación juntos para que la media móvil de los primeros días tenga su historia. La última columna es la mejora relativa sobre el mejor baseline. La consigna pide decir con honestidad qué se compra con la complejidad cuando esa mejora es chica, y esa discusión queda escrita en el ADR 0016.

In [17]:
en_validacion = evaluar_en_validacion(entrenamiento, validacion, columnas, variante_elegida, alpha_elegido)
mejor_en_pliegues = min(referencias["persistencia"].mean(), referencias["media_movil"].mean())
resumen_errores = pd.DataFrame({
    "Filas": [int(referencias["filas_evaluacion"].sum()), en_validacion["filas"]],
    "MAE persistencia": [referencias["persistencia"].mean(), en_validacion["persistencia"]],
    "MAE media móvil": [referencias["media_movil"].mean(), en_validacion["media_movil"]],
    f"MAE variante {variante_elegida}": [mae_elegida.mean(), en_validacion["mae"]],
    "Mejora sobre el mejor baseline (%)": [100 * (mejor_en_pliegues - mae_elegida.mean()) / mejor_en_pliegues,
                                           en_validacion["mejora_relativa"]]},
    index=["Pliegues (promedio de 5)", "Validación (un bloque, solo informa)"])
titulo_y_tabla(5, "Error de la variante elegida frente a los dos baselines", resumen_errores)

Tabla 7.5. Error de la variante elegida frente a los dos baselines


,Filas,MAE persistencia,MAE media móvil,MAE variante A,Mejora sobre el mejor baseline (%)
Pliegues (promedio de 5),677,3249.53,3376.83,3381.04,-4.05
"Validación (un bloque, solo informa)",171,2668.33,2497.65,2292.52,8.21


El objetivo de cada fila es el consumo del día siguiente, así que en la última fila de entrenamiento ese día ya pertenece a validación, y lo mismo pasa con la última fila de validación respecto de la prueba. Para que ninguna etiqueta cruce de un tramo al otro, esas dos etiquetas se dejan vacías y las filas se descartan del dataset final. Son dos filas de más de mil, así que se pierde muy poco. La tabla 7.6 muestra, con las filas alrededor de las dos fronteras, qué cambia: el valor de `Frio (Kw)` de hoy no se toca y solo desaparece el objetivo de las dos filas de borde.

In [18]:
sin_frontera = _quitar_etiquetas_de_frontera(tabla_final)
ultima_train, ultima_val = entrenamiento.index[-1], validacion.index[-1]
filas_borde = [ultima_train - 1, ultima_train, ultima_train + 1, ultima_val, ultima_val + 1]
tramo_de = lambda i: "entrenamiento" if i <= ultima_train else ("validación" if i <= ultima_val else "prueba")
frontera = pd.DataFrame({
    "Tramo": [tramo_de(i) for i in filas_borde],
    "DIA": tabla_final.loc[filas_borde, "DIA"].dt.date.astype(str),
    "Frio de hoy (kW)": tabla_final.loc[filas_borde, "Frio (Kw)"],
    "Objetivo antes (kW)": tabla_final.loc[filas_borde, "objetivo"],
    "Objetivo después (kW)": sin_frontera.loc[filas_borde, "objetivo"]}, index=filas_borde)
frontera.index.name = "Fila"
titulo_y_tabla(6, "Las dos etiquetas que cruzan de un tramo al siguiente (filas de borde)", frontera)

Tabla 7.6. Las dos etiquetas que cruzan de un tramo al siguiente (filas de borde)


,Tramo,DIA,Frio de hoy (kW),Objetivo antes (kW),Objetivo después (kW)
Fila,,,,,
846,entrenamiento,2022-10-27,24957.00,22048.00,22048.00
847,entrenamiento,2022-10-28,22048.00,17342.00,nan
848,validación,2022-10-29,17342.00,15813.00,15813.00
1029,validación,2023-04-27,21941.00,22115.00,nan
1030,prueba,2023-04-28,22115.00,17634.00,17634.00


Por último se ejecuta el cierre de la fase sin escribir archivos (los archivos reales los genera el script `src/preprocessing_pipeline.py`, que ya están versionados). El `Pipeline` se ajusta solo con `X_train`, y a `X_val` y `X_test` se les aplica la transformación aprendida con entrenamiento solo para comprobar que corre y para registrar sus dimensiones; nada de lo transformado se guarda, porque el dataset final va en las unidades originales. La tabla 7.7 compara las dimensiones antes y después del `Pipeline`, que coinciden porque la variante elegida no agrega columnas.

In [19]:
cierre = completar_fase2(preproceso, guardar=False)
dimensiones = pd.DataFrame({
    "Filas de entrada": {n: f[0] for n, f in cierre["shapes"].items()},
    "Columnas de entrada": {n: f[1] for n, f in cierre["shapes"].items()},
    "Filas tras el Pipeline": {n: f[0] for n, f in cierre["forma_tras_el_pipeline"].items()},
    "Columnas tras el Pipeline": {n: f[1] for n, f in cierre["forma_tras_el_pipeline"].items()}})
titulo_y_tabla(7, "Dimensiones de X_train, X_val y X_test antes y después del Pipeline", dimensiones, precision=0)

Tabla 7.7. Dimensiones de X_train, X_val y X_test antes y después del Pipeline


,Filas de entrada,Columnas de entrada,Filas tras el Pipeline,Columnas tras el Pipeline
X_train,830,14,830,14
X_val,176,14,176,14
X_test,178,14,178,14


La tabla 7.8 reúne los controles sobre los archivos que ya están guardados en `data/processed/`. Todos tienen que dar verdadero: que el dataset final está en las unidades originales, que no contiene las filas de frontera, que los tres tramos no se solapan en el tiempo, que la columna auxiliar `consumo_dia` coincide con `Frio (Kw)` (se guarda aparte porque la fase siguiente la necesita para recalcular los baselines, y no es una entrada del modelo), que la huella registrada en `data/checksums.json` corresponde al archivo, que las dimensiones del linaje coinciden con las de la tabla anterior y que el imputador del `Pipeline` usa la mediana del entrenamiento y no la de toda la tabla.

In [20]:
ruta_csv = RAIZ / "data/processed/dataset_final.csv"
guardado = pd.read_csv(ruta_csv, parse_dates=["DIA"])
linaje = json.loads((RAIZ / "data/processed/data_lineage.json").read_text(encoding="utf-8"))
huellas = json.loads((RAIZ / "data/checksums.json").read_text(encoding="utf-8"))
# Hay fechas repetidas en la tabla (ADR 0002), así que se compara por posición de fila y no por fecha:
# el CSV guarda, en orden, las filas con objetivo de la tabla sin las dos etiquetas de frontera.
esperado = sin_frontera.loc[sin_frontera["objetivo"].notna()]

# El imputador de la variante A se ajusta con entrenamiento (sin la fila de frontera) y debe dar su mediana
entren_f, _, _ = dividir_por_fecha(sin_frontera)
X_entren, _ = separar_xy(entren_f, columnas)
preprocesador = armar_pipeline(variante_elegida, columnas, aprender_reglas_de_grupos(X_entren, columnas)).named_steps["preprocessor"].fit(X_entren)
imputador = preprocesador.named_transformers_["numericas"].named_steps["imputar"]

por_tramo = guardado.groupby("tramo")["DIA"].agg(["min", "max"])
controles = pd.Series({
    "El CSV está en las unidades originales (Frio igual al de la tabla)":
        bool(np.allclose(guardado["Frio (Kw)"], esperado["Frio (Kw)"], equal_nan=True)),
    "El CSV tiene exactamente dos filas menos que las filas con objetivo de la tabla":
        len(guardado) == int(tabla_final["objetivo"].notna().sum()) - 2,
    "Los tres tramos no se solapan en el tiempo":
        bool(por_tramo.loc["entrenamiento", "max"] < por_tramo.loc["validacion", "min"]
             and por_tramo.loc["validacion", "max"] < por_tramo.loc["prueba", "min"]),
    "consumo_dia coincide con Frio (Kw)":
        bool(guardado["consumo_dia"].equals(guardado["Frio (Kw)"])),
    "La huella del CSV coincide con data/checksums.json":
        huella(ruta_csv) == huellas["processed/dataset_final.csv"],
    "Las dimensiones del linaje coinciden con las de la tabla 7.7":
        linaje["shapes"] == cierre["shapes"],
    "El imputador usa la mediana del entrenamiento":
        bool(np.allclose(imputador.statistics_, X_entren.median().to_numpy())),
    "El Pipeline elegido es la variante y el alpha registrados":
        (variante_elegida, alpha_elegido) == (VARIANTE_POR_DEFECTO, ALPHA_ELEGIDO)})
titulo_y_tabla(8, "Controles del dataset final (todos deben ser True)", controles.to_frame("Cumple"), precision=0)
assert controles.all(), controles[~controles]

Tabla 7.8. Controles del dataset final (todos deben ser True)


,Cumple
El CSV está en las unidades originales (Frio igual al de la tabla),True
El CSV tiene exactamente dos filas menos que las filas con objetivo de la tabla,True
Los tres tramos no se solapan en el tiempo,True
consumo_dia coincide con Frio (Kw),True
La huella del CSV coincide con data/checksums.json,True
Las dimensiones del linaje coinciden con las de la tabla 7.7,True
El imputador usa la mediana del entrenamiento,True
El Pipeline elegido es la variante y el alpha registrados,True


### Conclusión de la fase

Se eligieron 14 variables con un método de filtro basado en la correlación de Spearman con el objetivo, completadas por jerarquía con los componentes de las interacciones, y se comprobó que el `Pipeline` más simple dentro del límite de un desvío sigue siendo la mediana seguida del escalado estándar. El dataset final se guarda sin escalar, con una columna auxiliar para los baselines y sin las dos filas cuyo objetivo cruzaría de un tramo al siguiente. No se probó ningún imputador por vecinos adicional ni se usó la prueba para decidir nada.

Hay que tener presente tres límites. Los errores de entrenamiento son un promedio de cinco bloques distintos y los de validación son un único bloque, así que no se pueden comparar entre sí. Ridge solo se usó como juez: en los pliegues la variante elegida no supera con claridad a los baselines, y la mejora en validación, medida sobre el mejor de ellos, es moderada. Por último, la validación ya se usó para elegir el método de selección, de modo que no sirve como estimación independiente.

Lo que sigue es la fase de modelado: recalcular los baselines sobre el mismo conjunto que el modelo final, probar al menos cuatro modelos y decidir para cada uno qué preprocesamiento necesita, porque los modelos de árboles aceptan vacíos y no necesitan escalado.